# GRPO: group-relative rewards without a critic

**Model:** `LiquidAI/LFM2.5-350M` · **Training:** SVAMP official train · **Main test:** SVAMP · **Transfer test:** GSM8K

Group Relative Policy Optimization (GRPO) samples several answers to the same question and measures each reward relative to its group. These relative advantages replace PPO's learned critic. The actor still uses a clipped probability ratio; a frozen initial reference policy supplies an optional sampled KL penalty.

[DeepSeekMath](https://arxiv.org/abs/2402.03300) introduces GRPO. This tutorial implements the repository's compact binary-answer variant, not the full data, curriculum, reward shaping, or distributed training used by the paper. Four samples per prompt are the preset for the selected 350M model.

**Recipe:** Sample four answers per question → numeric rewards → normalize within each question group → clipped actor plus reference KL → fresh group rollout

You will inspect the raw and normalized dataset, verify the reward, work through the masked tensor shapes and exact objective, run real training, inspect its buffer, and evaluate seed and trained checkpoints. The small mathematical examples run on CPU; the full pretrained training stage needs an appropriate training environment. All five notebooks are independent recipes and import the same readable PyTorch implementation.

## 1. Environment and execution plan

From the repository root, install the training dependencies and Jupyter, then select that environment's kernel:

```bash
python -m pip install -r requirements-notebooks.txt
python -m ipykernel install --user --name self-distill-rl --display-name "Self-distillation RL"
python -m jupyter lab notebooks/
```

Use Python 3.10+ and a CUDA-compatible PyTorch build. The full target checkpoint is intended for CUDA training; there is no automatic quantization or LoRA conversion. The notebook defaults to the Transformers backend so one compatible environment suffices. For higher-throughput inference, install `requirements-rollout.txt` in a separate environment, set `BACKEND = "vllm"`, and point `ROLLOUT_PYTHON` at that environment's Python. vLLM's dependency pins can differ from the Transformers 5.x training environment. Both environments must be able to import this repository and access the same checkpoint files.

Actual generation, training, and evaluation run as separate processes. This releases generation/model memory between stages. The notebook kernel keeps only small demonstration tensors and JSON records; it never holds a full teacher alongside a training subprocess. Full logs are saved even though only their last lines are displayed.

Run cells in order. Dataset/model downloads need network access on the first run. Training and both full test benchmarks can take substantial time. Artifacts live under `artifacts/notebooks/<method>/`; rerunning with the same directory replaces its prior experiment files. Change `RUN_DIR` before comparing independent experiments. Outputs are cleared in the committed notebooks so no benchmark score is implied before you run them.

The configuration has an explicit `RL_NOTEBOOK_SMOKE=1` validation mode. It **requires** `RL_NOTEBOOK_MODEL` and `RL_NOTEBOOK_DATA_ROOT` pointing to a tiny local model and prepared toy train/test JSONL. It checks the real pipeline with tiny budgets and labels its results as smoke checks; it is not a pretrained-model benchmark. Leave it unset for the real recipe.

In [ ]:
import json
import os
import sys
from importlib import metadata
from pathlib import Path

import torch
from IPython.display import Markdown, display

# Works when Jupyter starts in either the repository root or notebooks/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "self_distill_rl").is_dir() and (p / "on_policy_loop.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Launch this notebook from the repository or its notebooks directory")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks"))
from recipe_utils import (
    evaluate_models,
    prepare_recipe_data,
    report_benchmarks,
    run_cli,
)

from self_distill_rl.io import causal_batch, read_jsonl
from self_distill_rl.presets import PRESETS

print("Repository:", ROOT, "| Python:", sys.executable, "| CUDA:", torch.cuda.is_available())

### What each configuration value controls

| Variable | Meaning |
|---|---|
| `ALGORITHM`, `PRESET`, `MODEL` | Method and selected checkpoint; shared presets supply its default model and limits |
| `DATASET`, `CROSS_DATASET` | Training/main-test benchmark and an additional arithmetic transfer benchmark |
| `DATA_ROOT`, `RUN_DIR` | Prepared source JSONL location and this experiment's artifacts |
| `SEED` | Training partition, sampling, and optimizer seeding; reproducibility still depends on hardware/backend |
| `TRAIN_LIMIT`, `VAL_LIMIT` | Number of official train questions used for fitting and disjoint validation |
| `ROUNDS` | Fresh generation → optimization cycles; optimizer state restarts each cycle |
| `MAX_TOKENS`, `MAX_SEQ_LENGTH` | Completion budget and total context limit, including private teacher context |
| `SAMPLES` | Answers per training question; one except GRPO's four |
| `ACCUMULATION`, `CHUNK_SIZE` | Microbatches per optimizer step and tokens per vocabulary projection chunk |
| `BACKEND`, `ROLLOUT_PYTHON` | Inference engine and its environment; training uses the notebook kernel's Python |
| `BENCHMARK_LIMIT` | `None` evaluates every available test question; an integer is explicitly a test subset |

The small training defaults make this a tutorial, not a recommended final training scale. To use nearly all training data, reserve validation first and set `TRAIN_LIMIT` to the remaining count. Tune using validation; only then run the official test cells. Keep model choice, response budget, decoding mode, and verifier fixed when comparing seed and trained checkpoints.

In [ ]:
ALGORITHM = "grpo"
PRESET = PRESETS[ALGORITHM]
SMOKE = os.environ.get("RL_NOTEBOOK_SMOKE") == "1"
if SMOKE and not (os.environ.get("RL_NOTEBOOK_MODEL") and os.environ.get("RL_NOTEBOOK_DATA_ROOT")):
    raise ValueError("Smoke mode requires a local tiny model and prepared toy data root")
MODEL = os.environ.get("RL_NOTEBOOK_MODEL", PRESET.model)
DATASET, CROSS_DATASET = PRESET.dataset, "gsm8k"
DATA_ROOT = Path(os.environ.get("RL_NOTEBOOK_DATA_ROOT", ROOT / "artifacts/datasets")).resolve()
RUN_DIR = Path(os.environ.get("RL_NOTEBOOK_RUN_ROOT", ROOT / "artifacts/notebooks")) / ALGORITHM
RUN_DIR = RUN_DIR.resolve()
RUN_DIR.mkdir(parents=True, exist_ok=True)
SEED, TRAIN_LIMIT, VAL_LIMIT, ROUNDS = 7, (2 if SMOKE else 64), (1 if SMOKE else 16), 1
MAX_TOKENS = 4 if SMOKE else PRESET.max_tokens
MAX_SEQ_LENGTH = 256 if SMOKE else PRESET.max_seq_length
SAMPLES = (2 if SMOKE else PRESET.samples_per_prompt) if ALGORITHM == "grpo" else 1
ACCUMULATION, CHUNK_SIZE = 4, 32
BACKEND, ROLLOUT_PYTHON = "transformers", sys.executable
BENCHMARK_LIMIT = 1 if SMOKE else None
CONFIG = {"algorithm": ALGORITHM, "model": MODEL, "dataset": DATASET, "cross_dataset": CROSS_DATASET,
              "seed": SEED, "train_limit": TRAIN_LIMIT, "validation_limit": VAL_LIMIT, "rounds": ROUNDS,
              "max_tokens": MAX_TOKENS, "max_seq_length": MAX_SEQ_LENGTH, "samples": SAMPLES,
              "accumulation": ACCUMULATION, "chunk_size": CHUNK_SIZE, "backend": BACKEND,
              "rollout_python": ROLLOUT_PYTHON, "benchmark_limit": BENCHMARK_LIMIT, "smoke": SMOKE,
              "versions": {name: metadata.version(name) for name in ("torch", "transformers", "datasets")},
              "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2) + "\n")
display(Markdown(f"**{'SMOKE CHECK ONLY' if SMOKE else 'Training recipe'}** — artifacts: `{RUN_DIR}`"))

## 2. Hugging Face datasets and detailed input format

We use [openai/gsm8k, config `main`](https://huggingface.co/datasets/openai/gsm8k) and [ChilleD/SVAMP, config `default`](https://huggingface.co/datasets/ChilleD/SVAMP). The prepared source sizes are:

| Dataset | Official train | Official test | Typical use here |
|---|---:|---:|---|
| GSM8K | 7,473 | 1,319 | Worked multi-step arithmetic; Qwen training and main evaluation |
| SVAMP | 700 | 300 | Short arithmetic word problems; LFM training and main evaluation |

GSM8K raw examples have `question` and `answer`, where `answer` contains a worked solution ending in `#### <number>`. SVAMP examples have `ID`, `Body`, `Question`, `Equation`, `Answer`, `Type`, and `question_concat`. We build its question from `Body + " " + Question`; `Equation` is useful as a compact reference. Additional raw metadata is not needed by the trainer.

Both sources become UTF-8 **JSONL**: one complete JSON object per line, not one large JSON array. Each object has these common fields:

| Field | Type | Exact role |
|---|---|---|
| `id` | string | Stable `<dataset>:<source split>:<original index or ID>` identity, assigned before shuffling |
| `prompt` | string | Question plus `Show concise reasoning. End with 'Final answer: <number>'.` |
| `answer` | string | Normalized numeric answer, not the whole worked solution |
| `reference` | string | GSM8K worked solution; SVAMP `Compute <Equation>. Final answer: <answer>` |
| `dataset` | string | Hugging Face repository ID identifying the source |
| `split` | string | Official `train` or `test`; reserved notebook validation becomes `validation` |

The following synthetic raw example demonstrates normalization; it is not represented as an actual downloaded benchmark row. The next preparation cell downloads the real official splits if missing, then prints an actual training record.

In [ ]:
from self_distill_rl.datasets import normalize_example

raw = {"ID": "demo", "Body": "There are 2 red and 3 blue balls.",
       "Question": "How many balls are there?", "Equation": "(2+3)",
       "Answer": 5, "Type": "Addition", "question_concat": "There are 2 red and 3 blue balls. How many balls are there?"}
example = normalize_example(DATASET, raw, "train", 0)
print("Synthetic raw row:\n", json.dumps(raw, indent=2))
print("Normalized row:\n", json.dumps(example, indent=2))

### Train, validation, and final test separation

The preparation helper seeds a shuffle of the official training pool, takes `TRAIN_LIMIT` examples for fitting, then takes `VAL_LIMIT` different examples for validation. Validation records retain their original IDs and add `source_split="train"` while setting `split="validation"`. A stable identity still indicates where the original row came from; the current `split` controls its evaluation role.

Only the training partition enters generation or optimization. The driver evaluates the seed and each round on **validation**, not official test. Main test prompts are checked against both train and validation. The second benchmark removes any exact prompt overlap with the used partitions and records how many were excluded; if any are removed, its result is a filtered test benchmark. Exact prompt checks do not establish freedom from paraphrase overlap or pretrained-model exposure.

`data_protocol.json` records partition IDs, source file SHA-256 hashes, preparation manifests, and test counts. For a reproducible dataset snapshot, prepare explicitly with `python prepare_datasets.py --revision <dataset-commit-sha>` before opening the notebook; `main` is otherwise a moving revision. Similarly, pin model revisions by downloading a specific snapshot and setting `MODEL` to its local path. Existing prepared JSONL is reused, so check its manifest and counts.

In [ ]:
paths = prepare_recipe_data(ROOT, DATA_ROOT, RUN_DIR, DATASET, TRAIN_LIMIT, VAL_LIMIT, SEED)
print({name: len(read_jsonl(path)) for name, path in paths.items()})
print("Actual training record:\n", json.dumps(read_jsonl(paths["train"])[0], indent=2))
print("Data provenance:", RUN_DIR / "data_protocol.json")

## 3. Numeric verification and its limits

The shared reward is

$$R(y,a)=\mathbf1[\operatorname{normalize}(\operatorname{extract}(y))=\operatorname{normalize}(a)].$$

The extractor recognizes a final-answer marker, GSM8K's `####`, boxed answers, and a fallback last number/line. Normalization handles decimals, fractions, thousands separators, and scientific notation using decimal/fraction arithmetic. This is numeric equivalence, not text similarity or an approximate floating-point tolerance. Closed `<think>...</think>` blocks are ignored when scoring; unfinished thinking has no accepted answer. Thinking is disabled by default for these short recipes.

The verifier does not prove every reasoning step, evaluate units, assess general chat behavior, or detect benchmark memorization. A last-number fallback can reward an accidental correct number, so the prompt requests an explicit `Final answer:` line. Preserve this same verifier across train, validation, seed evaluation, and trained evaluation. The small assertions below check the conventions before starting a costly run.

In [ ]:
from self_distill_rl.rewards import environment_feedback, exact_match_reward

checks = [("Final answer: 5", "5", 1.0), ("Final answer: 1/2", "0.5", 1.0),
          ("Final answer: 1,000", "1000", 1.0), ("Final answer: 4", "5", 0.0)]
for completion, expected, reward in checks:
    assert exact_match_reward(completion, expected) == reward
print("Verifier checks passed.")
print("Example feedback:", environment_feedback("Final answer: 4", "5"))

## 4. Tokens, causal shifting, and masks

Let $x$ denote the ordinary chat prompt and $y=(y_1,\ldots,y_T)$ its completion. An autoregressive policy factorizes as

$$\pi_\theta(y\mid x)=\prod_{t=1}^{T}\pi_\theta(y_t\mid x,y_{<t}).$$

The tokenizer renders the model's system/user chat template and an assistant generation prefix. The trainer concatenates prompt IDs and completion IDs, right-pads sequences, and computes:

| Tensor | Shape | Meaning |
|---|---|---|
| `input_ids` | $[B,S]$ | Prompt followed by completion, then padding |
| `attention_mask` | $[B,S]$ | Real input tokens, including both prompt and response |
| `action_mask` | $[B,S-1]$ | Positions whose next-token prediction belongs to the completion |
| Chosen log probabilities / values | $[B,S-1]$ | Shifted predictions; masked positions do not enter losses |
| Completion hidden states | list of $[T_i,H]$ | Decoder states that predict the sampled completion tokens |

At zero-based logit position $P-1$, the model predicts the **first** completion token when the prompt has length $P$. The action interval is therefore `[P-1 : P-1+T]`, not `[P : P+T]`. Prompt tokens help the model through attention but receive no direct response loss; padding receives neither attention nor loss. The toy integer IDs below make the shift visible without downloading a tokenizer.

In [ ]:
toy = causal_batch([[11, 12, 13], [11, 12]], [[21, 22], [23]], pad_token_id=0)
for name in ("input_ids", "attention_mask", "action_mask"):
    print(name, getattr(toy, name).tolist())
assert toy.action_mask.sum(dim=1).tolist() == [2, 1]
assert toy.action_mask[0].tolist() == [False, False, True, True]
print("Response-predicted token IDs:", toy.input_ids[:, 1:][toy.action_mask].tolist())

## 5. Group-relative advantages and the GRPO loss

For one question, sample $G=4$ responses with rewards $R_1,\ldots,R_G$. The code uses population standard deviation:

$$\mu=\frac1G\sum_jR_j,\qquad \sigma=\sqrt{\frac1G\sum_j(R_j-\mu)^2},\qquad
A_j=\begin{cases}(R_j-\mu)/(\sigma+10^{-8})&\sigma>10^{-8}\\0&\text{otherwise}.\end{cases}$$

Each response's scalar $A_j$ is broadcast to all its sampled tokens. Normalization must occur within one question group, not across unrelated prompts. A group with identical rewards supplies no actor advantage: all-wrong and all-correct groups both give zero. It can still contribute reference-KL loss when $\beta>0$.

As in PPO, $\rho_{jt}=\exp(\ell_{\theta,jt}-\ell_{\mathrm{old},jt})$. With a frozen initial reference $\pi_{\mathrm{ref}}$, let $d_{jt}=\ell_{\mathrm{ref},jt}-\ell_{\theta,jt}$. The nonnegative sampled penalty is $k_{jt}=e^{d_{jt}}-d_{jt}-1$:

$$\mathcal L_{\mathrm{GRPO}}=-\frac1B\sum_j\frac1{T_j}\sum_t
\left[\min\left(\rho_{jt}A_j,\operatorname{clip}(\rho_{jt},1-\epsilon,1+\epsilon)A_j\right)-\beta k_{jt}\right],\quad
\epsilon=0.2,\ \beta=0.01.$$

Here $B$ counts responses in the minibatch. Tokens are averaged **inside each response** before averaging responses, so long responses do not gain extra weight. This differs from a global valid-token mean. The trainer clamps log ratios and $d$ to $[-20,20]$ for stability.

When actions are sampled from the current policy, $k$ has expectation $\operatorname{KL}(\pi_\theta\|\pi_{\mathrm{ref}})$; in a reused behavior-policy buffer it is a surrogate, and clamping also changes the exact estimator. It is not an exact full-vocabulary KL evaluation. The reference checkpoint stays fixed across rounds. Setting `--beta 0` removes the penalty and avoids loading that model; there is never a value head.

In [ ]:
from self_distill_rl.objectives import group_relative_advantages, grpo_loss

rewards = torch.tensor([0.0, 1.0, 0.0, 1.0, 0.0, 0.0])
groups = ["question-A"] * 4 + ["question-B"] * 2
advantages = group_relative_advantages(rewards, groups)
assert torch.equal(advantages[4:], torch.zeros(2))
old_lp = torch.full((6, 2), -1.0)
new_lp = (old_lp + 0.05).requires_grad_()
mask = torch.ones_like(old_lp, dtype=torch.bool)
loss, metrics = grpo_loss(new_lp, old_lp, advantages, mask, 0.2, 0.01, old_lp)
loss.backward()
print("Group advantages:", advantages.tolist(), "loss metrics:", metrics)

## 6. Group identity is part of the dataset

GRPO uses the shared rollout schema below, with at least two responses per `group_id`. The preset generates four. All records in one group must have identical `prompt_token_ids`; the source question's ID is its group identity. The trainer batches **whole groups**, so `--groups-per-batch 1` contains four responses, not one response.

Never split a group across independent normalization batches, and never treat four unrelated questions as a group. The reference tokenizer must map vocabulary IDs identically to the student tokenizer. `reference_logprobs` is calculated by the frozen reference at training time; it is not the same quantity as the saved behavior-policy `old_logprobs`.

The optional textual `reference` and `feedback` fields are not fed to the actor or reference policy. GRPO uses ordinary question contexts and a scalar numeric reward. Because binary rewards often make all group outcomes equal, report the fraction of groups with mixed rewards; those are the groups that provide a nonzero policy-advantage signal.

### Shared on-policy rollout JSONL

For SDPO, OPSD, PPO, and GRPO, generation preserves the common dataset fields and adds:

| Field | Type / length | Purpose |
|---|---|---|
| `group_id` | string | Source question identity; same for all answers to that question |
| `prompt_token_ids` | list of $P$ integers | Exact rendered ordinary prompt supplied to generation |
| `completion` | string | Decoded answer for verifier/inspection |
| `completion_token_ids` | list of $T$ integers | Original generated actions, including sampled EOS if present |
| `old_logprobs` | list of $T$ finite floats | Log probability of each sampled token under the behavior policy |
| `reward` | finite float | Here 0 or 1 for normalized final-answer correctness |
| `feedback` | string | Verifier message, consumed by SDPO only |
| `model` | string | Checkpoint that generated this buffer; validated before training |
| `sampling` | object | `temperature` and `top_p` used for sampling |
| `enable_thinking` | Boolean | Must match the prompt rendering convention |
| `finish_reason` | string | `stop` or `length`; useful for truncation diagnostics |

For example, the **structural** fields might look like this (IDs/log probabilities are illustrative, not valid tokenizer output):

```json
{"group_id":"svamp:train:demo","prompt_token_ids":[11,12],"completion":"Final answer: 5","completion_token_ids":[21,22,23],"old_logprobs":[-0.7,-0.2,-1.1],"reward":1.0,"model":"<source checkpoint>","sampling":{"temperature":1.0,"top_p":1.0},"enable_thinking":false,"finish_reason":"stop"}
```

Never decode then re-encode a buffer to reconstruct actions; tokenization changes can invalidate likelihood ratios and prefix alignment. PPO and GRPO require **temperature 1, top-p 1, and top-k disabled** so stored likelihoods correspond to the unmodified policy distribution. No inherited repetition penalty or generation filter should silently change it. The shared generation code makes those choices explicitly and rejects incompatible policy-gradient buffers. Greedy evaluation is a separate protocol.

## 7. Model loading, memory, and optimization components

The code uses the model's actual architecture. Qwen3.5 is loaded through its multimodal auto class, trains its text backbone, freezes vision weights, and saves a native compatible checkpoint. LFM2.5 uses its causal-LM class. Both can use hybrid decoder operations, so simply replacing them with a generic Transformer block would change the model.

| Component | Implementation and reason |
|---|---|
| Trainable parameters | FP32 master weights so small full-parameter updates do not round away |
| Compute dtype | CUDA BF16 when supported, otherwise CUDA FP16 with gradient scaling; CPU FP32 |
| Attention | SDPA by default; optional eager or supported flash-attention paths |
| Checkpointing | Recompute intermediate activations during backward to trade compute for lower memory |
| Vocabulary head | Project only completion-predicting hidden states, in `CHUNK_SIZE` token chunks |
| Distribution losses | Exact vocabulary softmax/KL/JSD; chunking bounds temporary memory without truncating the vocabulary |
| Teacher/reference | Evaluation mode, no gradients or optimizer state; OPSD/GRPO keep an additional frozen model |
| Optimizer | FP32 AdamW, weight decay 0, global gradient-norm clipping at 1 |
| Accumulation | Average microbatch losses over `ACCUMULATION`; correctly rescale a shorter final window |
| Saved artifacts | Native policy config/weights/tokenizer; PPO additionally saves its scalar value head |

For microbatch loss $\mathcal L_k$, a full accumulation window applies

$$g=\frac1K\sum_{k=1}^{K}\nabla\mathcal L_k,\qquad g\leftarrow g\min\left(1,\frac1{\|g\|_2}\right),\qquad \theta\leftarrow\operatorname{AdamW}(\theta,g).$$

`K=ACCUMULATION` for a full window; a partial window uses its actual microbatch count. Unequal token counts can still make accumulation differ from one single global token-weighted batch because each microbatch has its own normalized loss. All optimization here is full-parameter training of the text policy, not adapters. Optimizer state is restarted at each round; weights and PPO's value head persist.

Start with the preset sequence limit, small batch size, checkpointing, and token chunks. A total limit must cover the ordinary prompt **and**, for self-distillation, the longer private teacher or rewriting prompt plus the response. Overlength data raises an error rather than silently dropping informative tokens. `MAX_TOKENS` controls generation length and therefore reward/truncation behavior; it is not just a memory setting.

### Components in this particular recipe

| Component | Shared source | What it does |
|---|---|---|
| Group sampling | `self_distill_rl.rollout` | Four independent completions per question with unmodified policy sampling |
| Reward verifier | `exact_match_reward` | Binary normalized numeric final-answer correctness |
| Complete groups | `group_batches` | Preserve same-question membership; validate size and identical prompt IDs |
| Relative signal | `group_relative_advantages` | Mean/population-standard-deviation normalization inside each group |
| Frozen reference | `load_policy(..., trainable=False)` | Initial checkpoint; compute chosen-token log probabilities without gradients |
| Loss and update | `grpo_loss`, `train_grpo.py` | Response-mean clipped surrogate; beta 0.01; one epoch; learning rate $10^{-6}$ |

Read [`train_grpo.py`](../train_grpo.py), [`modeling.py`](../self_distill_rl/modeling.py), [`io.py`](../self_distill_rl/io.py), and [`training.py`](../self_distill_rl/training.py) alongside the equations. These links expose the actual operations rather than a framework-specific trainer abstraction.

## 8. Run the actual generation → optimization recipe

The next cell calls the repository's real driver. Each round writes its generated training buffer, trains from the current checkpoint, saves the next checkpoint, and runs validation. The seed is evaluated on the same validation partition first. These are real subprocesses, not placeholder training functions.

The driver passes method-specific defaults to the small standalone trainer. To adjust a parameter outside the shared configuration, use the relevant trainer's `--help` and run the stages separately. This keeps the notebook compact while the equations and source links explain the operations being executed. Open the saved log for full per-batch diagnostics.

The last round is the predetermined checkpoint used below. If you instead select a checkpoint by validation accuracy, record that decision and use only that selected checkpoint for final test reporting. Do not select it by test accuracy.

In [ ]:
loop_root = RUN_DIR / "loop"
generation_args = ["--backend", BACKEND, "--max-tokens", MAX_TOKENS,
                   "--max-model-len", MAX_SEQ_LENGTH, "--max-num-seqs", 4,
                   "--gpu-memory-utilization", 0.75, "--seed", SEED]
training_seconds = run_cli(ROOT, ["on_policy_loop.py", "--algorithm", ALGORITHM,
    "--model", MODEL, "--prompts", paths["train"], "--eval-prompts", paths["validation"],
    "--train-limit", TRAIN_LIMIT, "--eval-limit", VAL_LIMIT, "--rounds", ROUNDS,
    "--output-root", loop_root, "--samples-per-prompt", SAMPLES,
    "--gradient-accumulation-steps", ACCUMULATION, "--logit-chunk-size", CHUNK_SIZE,
    "--rollout-python", ROLLOUT_PYTHON, *generation_args], RUN_DIR / "training.log")
LOOP_DIR = loop_root / ALGORITHM
FINAL_MODEL = LOOP_DIR / f"round_{ROUNDS:02d}_model"
assert (FINAL_MODEL / "config.json").exists()
print(f"Generation/training/validation wall time: {training_seconds:.1f}s; checkpoint: {FINAL_MODEL}")

## 9. Inspect the generated buffer and validation curve

Before trusting the score, inspect what actually entered the trainer. The first cell below reads the final round's training file and performs a method-specific check. The validation table and plot then compare the seed with every trained round using identical held-out questions and greedy decoding.

A tiny validation partition is useful for checking the pipeline but gives a noisy estimate. Increase it before model selection, and repeat independent training seeds when estimating algorithm variability. Validation loss/accuracy should guide changes to learning rate, epochs, response budget, or data—not the official test outputs in the next section.

In [ ]:
suffix = "sdft" if ALGORITHM == "sdft" else "rollouts"
buffer_path = LOOP_DIR / f"round_{ROUNDS:02d}_{suffix}.jsonl"
buffer = read_jsonl(buffer_path)
print("Generated training rows:", len(buffer), "| Fields:", sorted(buffer[0]))
from collections import defaultdict

by_group = defaultdict(list)
for row in buffer:
    by_group[row["group_id"]].append(row)
assert all(len(rows) == SAMPLES for rows in by_group.values())
mixed = sum(len({row["reward"] for row in rows}) > 1 for rows in by_group.values())
print(f"Groups with mixed rewards: {mixed}/{len(by_group)}")
print("First group's rewards:", [row["reward"] for row in next(iter(by_group.values()))])

Monitor mean reward, the fraction of mixed-reward groups, reference KL, and validation accuracy. If every group is all-wrong, merely taking more optimizer steps cannot create a policy-advantage signal. More samples, easier training questions, a warm-start supervised phase, or a different reward can change this, but each changes the experiment.

`train_grpo.py --help` exposes group batch size, policy epochs, clip epsilon, and beta. With one four-response group per minibatch and four accumulation steps, an optimizer window covers sixteen responses from four questions. Increasing group size changes memory and sampling cost. Keep the seed reference fixed when gathering later rounds.

In [ ]:
import matplotlib.pyplot as plt

labels = ["baseline"] + [f"round_{i:02d}" for i in range(1, ROUNDS + 1)]
validation_metrics = [json.loads((LOOP_DIR / f"{label}_eval.json").read_text()) for label in labels]
display(Markdown("| Checkpoint | Validation accuracy | Truncated |\n|---|---:|---:|\n" + "\n".join(
    f"| {label} | {row['accuracy']:.1%} | {row['truncation_rate']:.1%} |"
    for label, row in zip(labels, validation_metrics))))
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(labels, [row["accuracy"] for row in validation_metrics], marker="o")
ax.set(ylim=(-0.02, 1.02), ylabel="Greedy validation accuracy", title=f"{ALGORITHM.upper()} — validation")
fig.tight_layout()
fig.savefig(RUN_DIR / "validation_curve.png", dpi=150)
plt.show()
plt.close(fig)

## 10. Final official test and a second benchmark

The next cell compares **seed vs trained** on both official test datasets. The main benchmark is the same task family as training; the other dataset tests transfer to a different arithmetic distribution. Neither test file was used for rewrites, rollout training, feedback-based updates, or validation.

`BENCHMARK_LIMIT=None` requests all available official test rows: 1,319 GSM8K and 300 SVAMP before any recorded cross-dataset overlap removal. An integer requests a deterministic prefix subset and is labeled accordingly. Small smoke fixtures produce smoke results, not official benchmark scores. Downloading a partial source file earlier also limits the available test count; check `data_protocol.json` against the official sizes before claiming a full benchmark.

The fixed protocol uses one greedy answer per question, thinking disabled, identical response/context limits, the same chat prompt, and the same verifier for both checkpoints. Teacher references and verifier feedback are never supplied to the model during evaluation. These results are this repository's explicit protocol, not an asserted reproduction of a model card or leaderboard result. A second harder benchmark can expose response-budget limits; report truncation with accuracy.

For question rewards $R_i$ and $N$ evaluated questions:

$$\widehat{\mathrm{accuracy}}=\frac1N\sum_iR_i,\qquad
\mathrm{truncation\ rate}=\frac{\#\{\mathrm{finish}=\mathrm{length}\}}{N}.$$

The report includes mean completion tokens and a 95% Wilson interval for binary greedy correctness. It also counts paired gains/losses on identical question IDs; the net accuracy change is $(\mathrm{gained}-\mathrm{lost})/N$. Those intervals describe question-sample uncertainty under simplifying assumptions, not training-seed variability or proof of algorithm superiority. A negative change is a valid outcome; the notebook does not fabricate successful improvement.

Multi-sample evaluation is available in `evaluate.py`. Its `any_sample_accuracy` means at least one correct answer in the sampled set; it should not be compared directly to greedy accuracy or advertised as an unbiased pass@k estimate without the appropriate protocol.

In [ ]:
benchmark_results = evaluate_models(ROOT, RUN_DIR, ALGORITHM,
    models={"seed": MODEL, "trained": FINAL_MODEL},
    test_paths={name: paths[name] for name in (DATASET, CROSS_DATASET)},
    generation_args=generation_args, python=ROLLOUT_PYTHON, limit=BENCHMARK_LIMIT)
if SMOKE:
    for metrics in benchmark_results.values():
        metrics["scope"] = "SMOKE CHECK — toy data, not benchmark"
        output = RUN_DIR / f"benchmark_{metrics['dataset']}_{metrics['label']}.json"
        output.write_text(json.dumps(metrics, indent=2) + "\n")
paired_results = report_benchmarks(benchmark_results, RUN_DIR)

## 11. Inspect errors, preserve artifacts, and interpret results

The next cell reads the **saved** prediction records from the main benchmark. Each record contains question ID, sample index, prompt, expected answer, generated text, extracted answer, reward, token count, and finish reason. Reading the expected answer after evaluation is an analysis step; it did not enter the evaluation model context. Inspect some successes too: a correct final number does not guarantee a valid explanation.

Record your chosen hyperparameters and data/model revisions before running the final test. Test-error inspection is for reporting and planning a subsequent experiment; after making changes based on it, use a fresh assessment set if you want an unbiased final claim. Across algorithms, these presets use different model sizes and training distributions, so raw cross-notebook accuracy is not an isolated causal comparison of optimization methods.

Useful artifacts are `config.json` (configuration, package versions, device), `data_protocol.json` (IDs, hashes, manifests), `loop/<method>/round_XX_model/` (native checkpoint), the generated JSONL buffers, per-round validation metrics/predictions, `benchmark_<dataset>_<seed|trained>.json`, their `.predictions.jsonl` companions, `paired_comparison.json`, stage logs, and `validation_curve.png`. Logs include stage commands; subprocess wall time includes model loading and is not a pure token-throughput benchmark.

If CUDA runs out of memory, reduce the microbatch/group size where the trainer supports it, lower concurrency, and use smaller projection chunks while retaining checkpointing. Reducing sequence/response budgets changes the task and must be reported. If rollout provenance or group validation fails, regenerate from the intended checkpoint instead of editing metadata. If answer extraction fails, inspect explicit final-answer formatting and unfinished thinking. The notebooks make no guarantee that a short run improves the target models.

In [ ]:
predictions = read_jsonl(RUN_DIR / f"benchmark_{DATASET}_trained.predictions.jsonl")
errors = [row for row in predictions if row["reward"] == 0]
for row in (errors[:3] or predictions[:1]):
    print(json.dumps({key: row[key] for key in
        ("id", "prompt", "answer", "completion", "predicted_answer", "finish_reason")}, indent=2))
print(f"Saved {len(predictions)} main-test predictions and {len(errors)} errors.")

## References and next steps

[DeepSeekMath / GRPO paper](https://arxiv.org/abs/2402.03300) supplies the method's motivation. The equations above describe this repository's implementation and its explicitly stated adaptations. Dataset formats are documented in the [GSM8K card](https://huggingface.co/datasets/openai/gsm8k) and [SVAMP card](https://huggingface.co/datasets/ChilleD/SVAMP); architecture and tokenizer behavior follow the [LiquidAI/LFM2.5-350M model card](https://huggingface.co/LiquidAI/LFM2.5-350M).

For the rest of the repository, read the [algorithm guide](../docs/algorithms.md), [code walkthrough](../docs/code_walkthrough.md), and [notebook index](README.md). A useful next experiment changes one training variable, uses the validation partition to assess it, and keeps a fixed final test protocol. Report the actual data counts, source revisions, seed, decoding budget, and any deviations from this tutorial.